In [1]:
# ============================================================
# Results Notebook — Path Setup
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

# Recommended when the notebook is inside a notebooks/ folder:
# project_root/
#   notebooks/
#       results_tables_figures.ipynb
#   outputs/
#       data_audit/
#       feature_definition/
#       primary_statistics/
#       candidate_scorecard/
#       ...

PROJECT_ROOT = Path.cwd().parent

# Pipeline output directory
PIPELINE_OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Notebook-generated manuscript outputs
RESULTS_OUTPUT_DIR = PROJECT_ROOT / "results_outputs"
TABLE_DIR = RESULTS_OUTPUT_DIR / "tables"
FIGURE_DIR = RESULTS_OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Pipeline outputs:", PIPELINE_OUTPUT_DIR)
print("Notebook result outputs:", RESULTS_OUTPUT_DIR)

Project root: /Users/shivangayathri/Library/CloudStorage/OneDrive-Amritauniversity/MyStudents/Asha/Feature Detection for HI
Pipeline outputs: /Users/shivangayathri/Library/CloudStorage/OneDrive-Amritauniversity/MyStudents/Asha/Feature Detection for HI/outputs
Notebook result outputs: /Users/shivangayathri/Library/CloudStorage/OneDrive-Amritauniversity/MyStudents/Asha/Feature Detection for HI/results_outputs


In [2]:
# ============================================================
# Input CSV paths used across Results subsections
# ============================================================

# -----------------------------
# Cohort and ECG segment overview
# -----------------------------
data_integrity_path = (
    PIPELINE_OUTPUT_DIR / "data_audit" / "data_integrity_summary.csv"
)

segment_counts_path = (
    PIPELINE_OUTPUT_DIR / "data_audit" / "segment_counts_by_patient_condition.csv"
)

informative_feature_summary_path = (
    PIPELINE_OUTPUT_DIR / "feature_definition" / "informative_feature_summary.csv"
)

cohort_description_path = (
    PIPELINE_OUTPUT_DIR / "clinical_metadata" / "cohort_description_table.csv"
)

qc_exclusion_counts_path = (
    PIPELINE_OUTPUT_DIR / "segment_qc" / "qc_exclusion_counts.csv"
)

# -----------------------------
# Primary patient-level paired analysis
# -----------------------------
primary_paired_fdr_path = (
    PIPELINE_OUTPUT_DIR / "primary_statistics" / 
    "primary_paired_feature_results_all_features_fdr.csv"
)

fdr_summary_by_family_path = (
    PIPELINE_OUTPUT_DIR / "primary_statistics" / "fdr_summary_by_family.csv"
)

patient_feature_delta_path = (
    PIPELINE_OUTPUT_DIR / "patient_level" / 
    "patient_feature_delta_table_all_features.csv"
)

# -----------------------------
# Candidate scorecard
# -----------------------------
final_candidate_scorecard_path = (
    PIPELINE_OUTPUT_DIR / "candidate_scorecard" / "final_candidate_scorecard.csv"
)

final_candidate_categories_path = (
    PIPELINE_OUTPUT_DIR / "candidate_scorecard" / 
    "final_candidate_feature_categories.csv"
)

preliminary_priority_path = (
    PIPELINE_OUTPUT_DIR / "candidate_scorecard" / 
    "preliminary_feature_priority_table.csv"
)

preliminary_shortlist_path = (
    PIPELINE_OUTPUT_DIR / "candidate_scorecard" / 
    "preliminary_candidate_shortlist.csv"
)

# -----------------------------
# HRV-valid subgroup
# -----------------------------
hrv_valid_results_path = (
    PIPELINE_OUTPUT_DIR / "hrv_subgroup" / "hrv_valid_feature_results.csv"
)

hrv_valid_inclusion_path = (
    PIPELINE_OUTPUT_DIR / "hrv_subgroup" / 
    "hrv_valid_patient_inclusion_table.csv"
)

# -----------------------------
# Supportive mixed-effects models
# -----------------------------
lmm_results_path = (
    PIPELINE_OUTPUT_DIR / "mixed_effects" / 
    "lmm_supportive_results_all_features.csv"
)

# -----------------------------
# Sensitivity and robustness
# -----------------------------
sensitivity_summary_path = (
    PIPELINE_OUTPUT_DIR / "sensitivity" / 
    "sensitivity_vs_full_cohort_summary.csv"
)

robustness_matrix_path = (
    PIPELINE_OUTPUT_DIR / "sensitivity" / 
    "candidate_feature_robustness_matrix.csv"
)

# -----------------------------
# Patient-level agreement/disagreement
# -----------------------------
patient_disagreement_summary_path = (
    PIPELINE_OUTPUT_DIR / "agreement_disagreement" / 
    "patient_disagreement_summary.csv"
)

feature_agreement_summary_path = (
    PIPELINE_OUTPUT_DIR / "agreement_disagreement" / 
    "feature_agreement_summary.csv"
)

In [3]:
# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def get_metric(df, key, metric_col="metric", value_col="value"):
    """Return a scalar metric value from a two-column metric/value table."""
    match = df.loc[df[metric_col] == key, value_col]
    if match.empty:
        return np.nan
    return match.iloc[0]


def get_characteristic(df, key, characteristic_col="characteristic", value_col="value"):
    """Return a scalar value from a characteristic/value table."""
    match = df.loc[df[characteristic_col] == key, value_col]
    if match.empty:
        return np.nan
    return match.iloc[0]


def fmt_int(x):
    """Format integer-like values with commas."""
    if pd.isna(x):
        return ""
    return f"{int(float(x)):,}"

In [4]:
# ============================================================
# Shared IEEE-style figure settings
# ============================================================

IEEE_STYLE = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

plt.rcParams.update(IEEE_STYLE)

# Okabe-Ito colorblind-safe palette
COLORS = {
    "prehi": "#0072B2",      # blue
    "hi": "#E69F00",         # orange
    "increase": "#009E73",   # green
    "decrease": "#D55E00",   # vermillion
    "neutral": "#999999",    # grey
    "highlight": "#CC79A7",  # purple
}

# IEEE Access double-column width is approximately 7.16 inches
FIG_WIDTH_FULL = 7.16
FIG_WIDTH_SINGLE = 3.45

In [5]:
# ------------------------------------------------------------
# 4. Table I — Cohort, ECG Segment, and Analysis Design Summary
# ------------------------------------------------------------

# Total Pre-HI and During-HI segments
condition_segment_counts = (
    segment_counts
    .groupby("Condition_std", as_index=False)["n_segments"]
    .sum()
)

prehi_segments = int(
    condition_segment_counts.loc[
        condition_segment_counts["Condition_std"] == "PreHI", "n_segments"
    ].sum()
)

hi_segments = int(
    condition_segment_counts.loc[
        condition_segment_counts["Condition_std"] == "HI", "n_segments"
    ].sum()
)

# Number of patients with both conditions
patient_condition_counts = (
    segment_counts
    .groupby(["Patient_id", "Condition_std"])["n_segments"]
    .sum()
    .unstack(fill_value=0)
)

patients_with_both = int(
    ((patient_condition_counts.get("PreHI", 0) > 0) &
     (patient_condition_counts.get("HI", 0) > 0)).sum()
)

# Segment range per patient
patient_segment_totals = (
    segment_counts
    .groupby("Patient_id", as_index=False)["n_segments"]
    .sum()
)

segment_range = (
    f"{patient_segment_totals['n_segments'].min()}–"
    f"{patient_segment_totals['n_segments'].max()}"
)

# Episode count per patient, excluding PreHI label
hi_episode_counts = (
    segment_counts[segment_counts["Condition_std"] == "HI"]
    .groupby("Patient_id")["Episode_Label"]
    .nunique()
)

episode_range = f"{hi_episode_counts.min()}–{hi_episode_counts.max()}"

n_patients_more_than_one_hi_episode = int((hi_episode_counts > 1).sum())

table1 = pd.DataFrame({
    "Characteristic": [
        "Number of patients",
        "Total 2-min Lead-II ECG segments",
        "Pre-HI segments",
        "During-HI segments",
        "Patients with both Pre-HI and During-HI segments",
        "Segment count per patient, range",
        "HI episode labels per patient, range",
        "Patients with >1 HI episode label",
        "Full ECG-derived feature set",
        "Informative ECG-derived features analyzed",
        "Excluded non-informative features",
        "Minimum paired patients required per feature",
        "Primary statistical unit",
        "Primary paired summary",
        "Segment-level primary inference"
    ],
    "Value": [
        fmt_int(get_metric(data_integrity, "n_unique_patients")),
        fmt_int(get_metric(data_integrity, "n_rows")),
        fmt_int(prehi_segments),
        fmt_int(hi_segments),
        fmt_int(patients_with_both),
        segment_range,
        episode_range,
        fmt_int(n_patients_more_than_one_hi_episode),
        fmt_int(get_metric(informative_summary, "n_full_ecg_features")),
        fmt_int(get_metric(informative_summary, "n_informative_features")),
        fmt_int(get_metric(informative_summary, "n_excluded_features")),
        fmt_int(get_metric(informative_summary, "min_paired_patients_required")),
        "Patient",
        "One Pre-HI median and one During-HI median per patient-feature pair",
        "No"
    ]
})

table1

NameError: name 'segment_counts' is not defined

In [ ]:
# Save Table I as CSV and LaTeX
table1.to_csv(TABLE_DIR / "table1_cohort_segment_analysis_summary.csv", index=False)

latex_table1 = table1.to_latex(
    index=False,
    escape=False,
    column_format="p{0.55\\linewidth}p{0.35\\linewidth}",
    caption="Cohort, ECG Segment, and Analysis Design Summary.",
    label="tab:cohort_segment_analysis_summary"
)

with open(TABLE_DIR / "table1_cohort_segment_analysis_summary.tex", "w") as f:
    f.write(latex_table1)

print(latex_table1)

In [ ]:
# ------------------------------------------------------------
# 5. Table II — Clinical/Rhythm Context Summary
# ------------------------------------------------------------

clinical_rows_to_show = [
    "paced_or_icd",
    "BBB",
    "continuous_AF",
    "intermittent_or_new_AF",
    "any_af",
    "paced_or_bbb",
    "major_HRV_outlier",
    "antiarrhythmic_exposure",
    "mechanical_support",
    "atypical_HI",
    "clean_sinus_candidate",
    "exclude_from_HRV_valid_subgroup",
    "hrv_invalid_flag",
    "known_artifact_patient_flag",
    "high_confound",
    "moderate_or_high_confound",
    "confound_count_median_IQR"
]

clinical_label_map = {
    "paced_or_icd": "Paced rhythm or ICD",
    "BBB": "Bundle branch block",
    "continuous_AF": "Continuous atrial fibrillation",
    "intermittent_or_new_AF": "Intermittent or new atrial fibrillation",
    "any_af": "Any atrial fibrillation",
    "paced_or_bbb": "Pacing or bundle branch block",
    "major_HRV_outlier": "Major HRV/R-peak outlier",
    "antiarrhythmic_exposure": "Antiarrhythmic exposure",
    "mechanical_support": "Mechanical support",
    "atypical_HI": "Atypical HI context",
    "clean_sinus_candidate": "Clean sinus-rhythm candidate",
    "exclude_from_HRV_valid_subgroup": "Excluded from HRV-valid subgroup",
    "hrv_invalid_flag": "HRV-invalid flag",
    "known_artifact_patient_flag": "Known signal/artifact caution",
    "high_confound": "High clinical confound burden",
    "moderate_or_high_confound": "Moderate/high clinical confound burden",
    "confound_count_median_IQR": "Clinical confound count, median [IQR]"
}

table2_rows = []

for key in clinical_rows_to_show:
    value = get_characteristic(cohort_description, key)
    if pd.notna(value):
        table2_rows.append({
            "Clinical/Rhythm Characteristic": clinical_label_map.get(key, key),
            "Value": value
        })

table2 = pd.DataFrame(table2_rows)

table2

In [ ]:
# Save Table II as CSV and LaTeX
table2.to_csv(TABLE_DIR / "table2_clinical_rhythm_context_summary.csv", index=False)

latex_table2 = table2.to_latex(
    index=False,
    escape=False,
    column_format="p{0.58\\linewidth}p{0.25\\linewidth}",
    caption="Clinical and Rhythm Context of the 20-Patient Cohort.",
    label="tab:clinical_rhythm_context"
)

with open(TABLE_DIR / "table2_clinical_rhythm_context_summary.tex", "w") as f:
    f.write(latex_table2)

print(latex_table2)

In [ ]:
# ------------------------------------------------------------
# 6. Table III — Segment-QC Overview
# ------------------------------------------------------------

qc_label_map = {
    "qc_clear_segment_flag": "QC-clear segments",
    "clear_segment_qc_exclusion_flag": "Segments with clear QC exclusion flag",
    "near_zero_signal_flag": "Near-zero signal",
    "low_variability_signal_flag": "Low-variability signal",
    "qrs_std_zero_flag": "QRS standard deviation equal/near zero",
    "known_artifact_segment_flag": "Known artifact segment",
    "extreme_value_review_flag": "Extreme-value review flag",
    "extreme_kurtosis_review_flag": "Extreme kurtosis review flag",
    "known_patient_qc_caution_flag": "Known patient-level QC caution",
    "signal_review_flag": "Signal review flag",
    "missing_any_informative_feature_flag": "Missing any informative feature"
}

qc_table = qc_exclusion_counts.copy()
qc_table["QC category"] = qc_table["qc_flag"].map(qc_label_map).fillna(qc_table["qc_flag"])
qc_table["Segments, n"] = qc_table["n_segments_flagged"].astype(int)
qc_table["Segments, %"] = qc_table["percent_segments_flagged"].map(lambda x: f"{x:.1f}")

table3 = qc_table[["QC category", "Segments, n", "Segments, %"]].copy()

table3

In [ ]:
# Save Table III as CSV and LaTeX
table3.to_csv(TABLE_DIR / "table3_segment_qc_overview.csv", index=False)

latex_table3 = table3.to_latex(
    index=False,
    escape=False,
    column_format="p{0.58\\linewidth}rr",
    caption="Segment-Level Quality-Control Overview. QC flags were used for review and sensitivity analysis, not for primary full-cohort exclusion.",
    label="tab:segment_qc_overview"
)

with open(TABLE_DIR / "table3_segment_qc_overview.tex", "w") as f:
    f.write(latex_table3)

print(latex_table3)

In [ ]:
# ------------------------------------------------------------
# 7. Figure 1 — Patient-wise ECG segment availability
# ------------------------------------------------------------

# Collapse across episode labels to total Pre-HI and HI segments per patient
segment_by_patient_condition = (
    segment_counts
    .groupby(["Patient_id", "Condition_std"], as_index=False)["n_segments"]
    .sum()
)

segment_wide = (
    segment_by_patient_condition
    .pivot(index="Patient_id", columns="Condition_std", values="n_segments")
    .fillna(0)
)

# Ensure consistent columns
for col in ["PreHI", "HI"]:
    if col not in segment_wide.columns:
        segment_wide[col] = 0

segment_wide = segment_wide[["PreHI", "HI"]].copy()

# Sort by patient ID numerically when possible
segment_wide = segment_wide.reset_index()
segment_wide["Patient_id_numeric"] = pd.to_numeric(segment_wide["Patient_id"], errors="coerce")
segment_wide = segment_wide.sort_values(["Patient_id_numeric", "Patient_id"])
segment_wide = segment_wide.drop(columns=["Patient_id_numeric"])

segment_wide

In [ ]:
# Plot
# ------------------------------------------------------------
# 2. Shared IEEE-style figure settings
# Reuse this block for later manuscript figures.
# ------------------------------------------------------------

IEEE_STYLE = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

# Okabe–Ito colorblind-safe palette
COLOR_PREHI = "#0072B2"   # blue
COLOR_HI = "#E69F00"      # orange

plt.rcParams.update(IEEE_STYLE)

# IEEE Access double-column width is approximately 7.16 inches.
FIG_WIDTH_IN = 7.16
FIG_HEIGHT_IN = 3.2

# ------------------------------------------------------------
# 4. Figure function
# ------------------------------------------------------------

def plot_segment_availability(segment_wide, save_dir, filename_stem):
    """
    Plot patient-wise ECG segment availability as a stacked bar chart.

    Parameters
    ----------
    segment_wide : pandas.DataFrame
        DataFrame with columns ["Patient_id", "PreHI", "HI"].
    save_dir : pathlib.Path
        Output directory for figure files.
    filename_stem : str
        Base filename without extension.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Generated figure object.
    """

    required_cols = {"Patient_id", "PreHI", "HI"}
    missing_cols = required_cols - set(segment_wide.columns)

    if missing_cols:
        raise ValueError(f"segment_wide is missing columns: {missing_cols}")

    x = np.arange(len(segment_wide))
    bar_width = 0.72

    fig, ax = plt.subplots(
        figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
        constrained_layout=True
    )

    ax.bar(
        x,
        segment_wide["PreHI"],
        width=bar_width,
        label="Pre-HI",
        color=COLOR_PREHI,
        edgecolor="white",
        linewidth=0.3,
    )

    ax.bar(
        x,
        segment_wide["HI"],
        width=bar_width,
        bottom=segment_wide["PreHI"],
        label="During-HI",
        color=COLOR_HI,
        edgecolor="white",
        linewidth=0.3,
    )

    ax.set_xlabel("Patient ID")
    ax.set_ylabel("Number of 2-minute ECG segments")

    ax.set_xticks(x)
    ax.set_xticklabels(
        segment_wide["Patient_id"].astype(str),
        rotation=45,
        ha="right",
    )

    ax.grid(
        axis="y",
        linestyle=":",
        linewidth=0.6,
        color="0.6",
        zorder=0
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        frameon=False,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        handlelength=1.2,
        columnspacing=1.2,
    )

    save_dir.mkdir(parents=True, exist_ok=True)

    fig.savefig(
        save_dir / f"{filename_stem}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        save_dir / f"{filename_stem}.png",
        dpi=600,
        bbox_inches="tight"
    )

    return fig


In [ ]:
fig = plot_segment_availability(
    segment_wide=segment_wide,
    save_dir=FIGURE_DIR,
    filename_stem="figure1_patient_wise_segment_availability"
)

plt.show()

In [ ]:
# ------------------------------------------------------------
# 8. Suggested caption text
# ------------------------------------------------------------

figure1_caption = (
    "Patient-wise availability of non-overlapping 2-minute Lead-II ECG segments "
    "in the Pre-HI and During-HI conditions. Segment availability varied across "
    "patients because the number and duration of clinically labelled HI episodes "
    "differed across the cohort. The primary analysis therefore collapsed "
    "segment-level feature values into paired patient-level medians before "
    "inferential testing."
)

print(figure1_caption)